In [13]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

# ========== 修正文件路径（适配你本地存放位置） ==========
candidates = [
    Path(r"C:\Users\Administrator\EDIT-1-24012456\day3_pandas\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("../data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请检查文件存放路径")

# 项目输出目录
OUTPUT_DIR = Path("output/day04_project")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 读取原始数据
raw_df = pd.read_excel(DATA_PATH, sheet_name="E Comm")

print(f"原始数据文件：{DATA_PATH}")
print(f"输出目录：{OUTPUT_DIR.resolve()}")
print(f"原始数据形状：{raw_df.shape[0]} 行 , {raw_df.shape[1]} 列")
raw_df.head()


原始数据文件：..\data\E Commerce Dataset.xlsx
输出目录：C:\Users\Administrator\EDIT-1-24012456\day4_pandas2\output\day04_project
原始数据形状：5630 行 , 20 列


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


In [14]:
# 1.每条记录代表什么？
# 单条电商平台用户完整画像，包含会员时长、设备、支付、订单、流失状态等全维度行为数据
# 2.项目的目标变量是哪一列？
# Churn：1代表用户流失，0代表用户留存，是流失预测目标字段
# 3.为什么 CustomerID 不应作为普通连续数值参与后续分析？
# CustomerID仅为用户唯一标识，无数值大小意义，不能做加减运算，属于分类唯一编号，直接当数值会误导模型

In [15]:
def build_quality_report(data):
    """返回字段级数据质量报告：类型、缺失数量、缺失比例、唯一值数量"""
    report = pd.DataFrame()
    report["字段类型"] = data.dtypes
    report["缺失数量"] = data.isna().sum()
    report["缺失比例(%)"] = round(data.isna().mean() * 100, 2)
    report["唯一值数量"] = data.nunique()
    return report

# 生成清洗前质量报告
quality_before = build_quality_report(raw_df)
display(quality_before)

,字段类型,缺失数量,缺失比例(%),唯一值数量
CustomerID,int64,0,0.00,5630
Churn,int64,0,0.00,2
Tenure,float64,264,4.69,36
PreferredLoginDevice,str,0,0.00,3
CityTier,int64,0,0.00,3
WarehouseToHome,float64,251,4.46,34
PreferredPaymentMode,str,0,0.00,7
Gender,str,0,0.00,2
HourSpendOnApp,float64,255,4.53,6
NumberOfDeviceRegistered,int64,0,0.00,6


In [16]:
# 完全重复整行
total_dup = raw_df.duplicated().sum()
# 用户ID重复
cid_dup = raw_df["CustomerID"].duplicated().sum()
# 流失分布
churn_cnt = raw_df["Churn"].value_counts()
churn_rate = round(raw_df["Churn"].mean() * 100, 2)

print("完全重复行数：", total_dup)
print("CustomerID 重复数量：", cid_dup)
print("\nChurn 流失分布：")
print(churn_cnt)
print(f"整体用户流失率：{churn_rate}%")

# 分类字段频数
cat_list = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in cat_list:
    print(f"\n===== {col} =====")
    print(raw_df[col].value_counts())

完全重复行数： 0
CustomerID 重复数量： 0

Churn 流失分布：
Churn
0    4682
1     948
Name: count, dtype: int64
整体用户流失率：16.84%

===== PreferredLoginDevice =====
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

===== PreferredPaymentMode =====
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

===== PreferedOrderCat =====
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


In [17]:
NUMERIC_MISSING_COLS = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

CATEGORY_MAPPINGS = {
    "PreferredLoginDevice": {
        "Phone": "Mobile Phone",
        "Mobile": "Mobile Phone"
    },
    "PreferredPaymentMode": {
        "COD": "Cash on Delivery",
        "CC": "Credit Card"
    },
    "PreferedOrderCat": {
        "Mobile": "Mobile Phone"
    }
}

In [18]:
def clean_ecommerce_data(data):
    """
    清洗电商用户行为数据。
    参数：data: 原始用户行为 DataFrame
    返回：cleaned_df: 清洗后数据集；cleaning_log: 处理日志表
    """
    # 复制数据，不修改原始df
    df = data.copy()
    logs = []
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 步骤1：删除完全重复行
    before_row = df.shape[0]
    df = df.drop_duplicates()
    after_row = df.shape[0]
    affect = before_row - after_row
    logs.append({
        "处理时间": now,
        "步骤": "删除完全重复行",
        "规则": "移除完全一致的重复记录",
        "处理前行数": before_row,
        "处理后行数": after_row,
        "影响记录数": affect
    })

    # 步骤2：数值字段中位数填充缺失
    for col in NUMERIC_MISSING_COLS:
        miss_before = df[col].isna().sum()
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        miss_after = df[col].isna().sum()
        affect = miss_before - miss_after
        logs.append({
            "处理时间": now,
            "步骤": f"中位数填充-{col}",
            "规则": f"使用字段中位数 {med_val:.2f} 填补缺失",
            "处理前行数": miss_before,
            "处理后行数": miss_after,
            "影响记录数": affect
        })

    # 步骤3：分类字段统一名称
    for col, map_dict in CATEGORY_MAPPINGS.items():
        for old_val, new_val in map_dict.items():
            cnt = (df[col] == old_val).sum()
            df[col] = df[col].replace(old_val, new_val)
            logs.append({
                "处理时间": now,
                "步骤": f"类别标准化-{col}",
                "规则": f"将 {old_val} 统一映射为 {new_val}",
                "处理前行数": cnt,
                "处理后行数": 0,
                "影响记录数": cnt
            })

    # 步骤4：目标字段转为整型
    df["Churn"] = df["Churn"].astype(int)
    df["Complain"] = df["Complain"].astype(int)

    # 生成日志DataFrame
    cleaning_log = pd.DataFrame(logs)
    return df, cleaning_log

# 执行清洗
cleaned_df, cleaning_log = clean_ecommerce_data(raw_df)
display(cleaning_log)
cleaned_df.head()

,处理时间,步骤,规则,处理前行数,处理后行数,影响记录数
0,2026-07-21 10:04:58,删除完全重复行,移除完全一致的重复记录,5630,5630,0
1,2026-07-21 10:04:58,中位数填充-Tenure,使用字段中位数 9.00 填补缺失,264,0,264
2,2026-07-21 10:04:58,中位数填充-WarehouseToHome,使用字段中位数 14.00 填补缺失,251,0,251
3,2026-07-21 10:04:58,中位数填充-HourSpendOnApp,使用字段中位数 3.00 填补缺失,255,0,255
4,2026-07-21 10:04:58,中位数填充-OrderAmountHikeFromlastYear,使用字段中位数 15.00 填补缺失,265,0,265
5,2026-07-21 10:04:58,中位数填充-CouponUsed,使用字段中位数 1.00 填补缺失,256,0,256
6,2026-07-21 10:04:58,中位数填充-OrderCount,使用字段中位数 2.00 填补缺失,258,0,258
7,2026-07-21 10:04:58,中位数填充-DaySinceLastOrder,使用字段中位数 3.00 填补缺失,307,0,307
8,2026-07-21 10:04:58,类别标准化-PreferredLoginDevice,将 Phone 统一映射为 Mobile Phone,1231,0,1231
9,2026-07-21 10:04:58,类别标准化-PreferredLoginDevice,将 Mobile 统一映射为 Mobile Phone,0,0,0


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


In [19]:
def iqr_outlier_summary(series):
    """输出 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return {
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "下限": round(lower, 2),
        "上限": round(upper, 2),
        "候选异常值数量": int(((series < lower) | (series > upper)).sum())
    }

# 1. 新增用户时长分层 TenureGroup
bins = [0, 12, 24, 36, 1000]
labels = ["0-12个月", "13-24个月", "25-36个月", "36个月以上"]
cleaned_df["TenureGroup"] = pd.cut(cleaned_df["Tenure"], bins=bins, labels=labels)

# 2. 新增移动端登录标记 IsMobileLogin
mobile_tags = ["Mobile Phone"]
cleaned_df["IsMobileLogin"] = np.where(cleaned_df["PreferredLoginDevice"].isin(mobile_tags), 1, 0)

# 3. 生成异常值报告
outlier_cols = ["WarehouseToHome", "OrderCount", "CashbackAmount"]
outlier_report = []
for c in outlier_cols:
    res = iqr_outlier_summary(cleaned_df[c])
    res["字段名"] = c
    outlier_report.append(res)
outlier_report = pd.DataFrame(outlier_report)
print("===== IQR候选异常值报告 =====")
display(outlier_report)

===== IQR候选异常值报告 =====


,Q1,Q3,下限,上限,候选异常值数量,字段名
0,9.00,20.00,-7.50,36.50,2,WarehouseToHome
1,1.00,3.00,-2.00,6.00,703,OrderCount
2,145.77,196.39,69.84,272.33,438,CashbackAmount


In [20]:
# 统计不合规数据
rule_data = [
    ["使用时长小于0", (cleaned_df["Tenure"] < 0).sum()],
    ["仓库距离小于0", (cleaned_df["WarehouseToHome"] < 0).sum()],
    ["订单数小于或等于0", (cleaned_df["OrderCount"] <= 0).sum()],
    ["返现金额小于0", (cleaned_df["CashbackAmount"] < 0).sum()]
]
business_rule_report = pd.DataFrame(rule_data, columns=["规则", "不合规记录数"])
display(business_rule_report)

# 处理结论
# 本数据集无负数违规业务数据，无需删除/修正；若存在负数，需和业务确认是录入错误还是特殊标记，不可直接丢弃

,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数小于或等于0,0
3,返现金额小于0,0


In [21]:
# 清洗后质量报告
quality_after = build_quality_report(cleaned_df)

# 验收断言
assert cleaned_df[NUMERIC_MISSING_COLS].isna().sum().sum() == 0, "数值字段仍存在缺失值"
assert "Phone" not in cleaned_df["PreferredLoginDevice"].unique(), "登录设备未完成标准化"
assert "COD" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式简写未清理"
assert "CC" not in cleaned_df["PreferredPaymentMode"].unique(), "支付方式简写未清理"
assert {"TenureGroup", "IsMobileLogin"}.issubset(cleaned_df.columns), "衍生特征缺失"

# 导出4份交付文件
quality_before.to_csv(OUTPUT_DIR / "data_quality_before.csv", index=True, encoding="utf-8-sig")
quality_after.to_csv(OUTPUT_DIR / "data_quality_after.csv", index=True, encoding="utf-8-sig")
cleaning_log.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")
cleaned_df.to_csv(OUTPUT_DIR / "ecommerce_customer_cleaned.csv", index=False, encoding="utf-8-sig")

print("✅ 全部校验通过！")
print(f"清洗前质量报告：{OUTPUT_DIR / 'data_quality_before.csv'}")
print(f"清洗后质量报告：{OUTPUT_DIR / 'data_quality_after.csv'}")
print(f"数据处理日志：{OUTPUT_DIR / 'cleaning_log.csv'}")
print(f"最终清洗数据集：{OUTPUT_DIR / 'ecommerce_customer_cleaned.csv'}")
print("\n异常值报告：")
display(outlier_report)
print("\n业务规则校验报告：")
display(business_rule_report)

✅ 全部校验通过！
清洗前质量报告：output\day04_project\data_quality_before.csv
清洗后质量报告：output\day04_project\data_quality_after.csv
数据处理日志：output\day04_project\cleaning_log.csv
最终清洗数据集：output\day04_project\ecommerce_customer_cleaned.csv

异常值报告：


,Q1,Q3,下限,上限,候选异常值数量,字段名
0,9.00,20.00,-7.50,36.50,2,WarehouseToHome
1,1.00,3.00,-2.00,6.00,703,OrderCount
2,145.77,196.39,69.84,272.33,438,CashbackAmount



业务规则校验报告：


,规则,不合规记录数
0,使用时长小于0,0
1,仓库距离小于0,0
2,订单数小于或等于0,0
3,返现金额小于0,0


In [22]:
#数据问题：数值字段存在缺失、分类名称不统一、存在统计学极值异常值；
#处理策略：缺失值用中位数填充、同义类别统一命名、异常值仅记录不删除；
#可用于后续分析：无缺失、分类标准化、新增分层特征，数据干净无噪声；
#需业务确认：IQR 筛选出的极值是否为真实高价值用户，负数业务阈值定义。